# 🧪 W12-D3 概念实验：为什么 MallSenseAI 选择了 YOLO-World？

> 配套阅读：`第12周-Day3-Detection体系-YOLO-World选型.md`（选型矩阵和 ADR 依据在那边）
> 这个 notebook 用三个可执行实验验证当天三个核心论断：
>
> 1. **闭集模型天生看不见 COCO 外的世界**——词表错配是结构性的，不是精度问题
> 2. **ROI 几何过滤是零样本精度不足的管线补偿**——两道几何闸门如何工作
> 3. **误报率是系统属性不是模型属性**——四层兜底把"弱模型"救成"可用系统"
>
> 实验环境：纯 numpy 模拟（语义空间为手工构造的教学近似），不加载真实模型权重。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

rng = np.random.default_rng(42)
print("字体就绪:", font_name)

## 实验 1：闭集 vs 开放词表——词表错配是结构性的

用一个 6 维手工语义空间模拟"视觉特征"：`[有轮, 容器, 手提, 大件, 可燃, 家具]`。

- **闭集检测器**（yolo11n/COCO）：固定 80 类 → 这里取 8 个 COCO 类做权重矩阵，输出只能是这 8 个标签之一
- **开放词表检测器**（YOLO-World）：候选类向量可运行时替换，打分 = 余弦相似度

看同一个"购物车"进两个检测器，发生什么。

In [ ]:
# 6 维手工语义空间: [有轮, 容器, 手提, 大件, 可燃, 家具]
DIMS = ["有轮", "容器", "手提", "大件", "可燃", "家具"]

# 闭集词表：COCO 80 类里的 8 个代表（权重"烧死"在模型里）
COCO_CLASSES = {
    "suitcase":  [0, 1, 1, 1, 0, 0],
    "backpack":  [0, 1, 1, 0, 0, 0],
    "umbrella":  [0, 0, 1, 0, 0, 0],
    "bicycle":   [1, 0, 0, 1, 0, 0],
    "chair":     [0, 0, 0, 1, 0, 1],
    "bottle":    [0, 1, 0, 0, 1, 0],
    "bench":     [0, 0, 0, 1, 0, 1],
    "handbag":   [0, 1, 1, 0, 0, 0],
}

# 开放词表：MallSenseAI DEFAULT_WORLD_CLASSES 的子集（运行时 set_classes 可随意增删）
WORLD_CLASSES = {
    "shopping cart": [1, 1, 0, 1, 0, 0],
    "stroller":      [1, 1, 0, 1, 0, 0],
    "wheelchair":    [1, 0, 0, 1, 0, 0],
    "traffic cone":  [0, 0, 0, 1, 1, 0],
    "cleaning cart": [1, 1, 0, 1, 0, 0],
    "ladder":        [0, 0, 0, 1, 0, 0],
}

# 测试目标：商场真实障碍物（全部不在 COCO 80 类里）
TARGETS = {
    "购物车": [1, 1, 0, 1, 0, 0],
    "交通锥": [0, 0, 0, 1, 1, 0],
    "梯子":   [0, 0, 0, 1, 0, 0],
}

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

def closed_vocab_detect(x, temperature=6.0):
    """闭集模型：logits = 类权重·特征，softmax 后 argmax。标签只可能来自固定词表。"""
    names = list(COCO_CLASSES)
    W = np.array([COCO_CLASSES[n] for n in names], dtype=float)
    probs = softmax(W @ np.array(x) * temperature)
    order = np.argsort(-probs)
    return [(names[i], probs[i]) for i in order[:3]]

def open_vocab_detect(x, candidates):
    """开放词表：与候选类向量算余弦相似度，候选集可运行时替换（= set_classes）。"""
    x = np.array(x, dtype=float)
    scores = {}
    for name, vec in candidates.items():
        v = np.array(vec, dtype=float)
        scores[name] = float(x @ v / (np.linalg.norm(x) * np.linalg.norm(v) + 1e-9))
    return sorted(scores.items(), key=lambda kv: -kv[1])[:3]

for target_name, x in TARGETS.items():
    closed_top = closed_vocab_detect(x)
    open_top = open_vocab_detect(x, WORLD_CLASSES)
    print(f"真实目标【{target_name}】特征={x}")
    print(f"  闭集模型 yolo11n 输出: " + ", ".join(f"{n}({p:.2f})" for n, p in closed_top))
    print(f"  开放词表 YOLO-World 输出: " + ", ".join(f"{n}({s:.2f})" for n, s in open_top))
    print()

In [ ]:
# 开放词表的"超能力"：换词表 = 换候选矩阵，模型本身不动一根毫毛
# 模拟运营改 detector_configs：把 ladder 换成 power bank（共享充电宝）
NEW_CLASSES = dict(WORLD_CLASSES)
NEW_CLASSES.pop("ladder")
NEW_CLASSES["power bank"] = [0, 1, 1, 0, 1, 0]   # 小、盒装、可燃(锂电池)——纯文本定义，零训练

x = TARGETS["梯子"]
print("检测目标改为 power bank 后，同一个'梯子'视觉特征：")
print("  旧词表:", ", ".join(f"{n}({s:.2f})" for n, s in open_vocab_detect(x, WORLD_CLASSES)))
print("  新词表:", ", ".join(f"{n}({s:.2f})" for n, s in open_vocab_detect(x, NEW_CLASSES)))
print()
print("→ 闭集模型要做到这一点必须重新训练发版；")
print("→ YOLO-World 只需要 UPDATE detector_configs 一行 SQL，10 秒内热生效。")
print("→ 这就是'检测什么'从权重(代码域)变成配置(运营域)。")

## 实验 2：ROI 几何过滤——零样本精度不足的管线补偿

`yolo_world.py` 的真实逻辑：`min_confidence=0.25`（比火灾检测低一半，先放进来），
然后两道几何闸门：
1. **质心必须在 ROI 多边形内**（射线法判断点在多边形内）
2. **bbox∩ROI 面积比 ≥ min_area_ratio（默认 0.005）**

用纯 numpy 实现这两道闸门，看它们如何把误报拦在规则引擎之前。

In [ ]:
def point_in_polygon(px, py, poly):
    """射线法：点是否在多边形内（归一化坐标）。"""
    n, inside = len(poly), False
    j = n - 1
    for i in range(n):
        xi, yi = poly[i]; xj, yj = poly[j]
        if (yi > py) != (yj > py) and px < (xj - xi) * (py - yi) / (yj - yi) + xi:
            inside = not inside
        j = i
    return inside

def clip_polygon(subject, clip):
    """Sutherland-Hodgman 凸多边形裁剪（ROI 取凸四边形，与算法假设一致）。"""
    def inside(p, a, b):
        return (b[0]-a[0])*(p[1]-a[1]) - (b[1]-a[1])*(p[0]-a[0]) >= 0
    output = subject[:]
    for i in range(len(clip)):
        a, b = clip[i], clip[(i+1) % len(clip)]
        input_list, output = output, []
        if not input_list: break
        prev = input_list[-1]
        for cur in input_list:
            if inside(cur, a, b):
                if not inside(prev, a, b):
                    output.append(_intersect(prev, cur, a, b))
                output.append(cur)
            elif inside(prev, a, b):
                output.append(_intersect(prev, cur, a, b))
            prev = cur
    return output

def _intersect(p1, p2, a, b):
    d1 = (b[0]-a[0])*(p1[1]-a[1]) - (b[1]-a[1])*(p1[0]-a[0])
    d2 = (b[0]-a[0])*(p2[1]-a[1]) - (b[1]-a[1])*(p2[0]-a[0])
    t = d1 / (d1 - d2 + 1e-12)
    return (p1[0] + t*(p2[0]-p1[0]), p1[1] + t*(p2[1]-p1[1]))

def polygon_area(poly):
    s = 0.0
    for i in range(len(poly)):
        x1, y1 = poly[i]; x2, y2 = poly[(i+1) % len(poly)]
        s += x1*y2 - x2*y1
    return abs(s) / 2

def yolo_world_gates(bbox, roi, min_area_ratio=0.005):
    """复刻 yolo_world.py 的两道几何闸门。bbox=(x1,y1,x2,y2) 归一化坐标。"""
    x1, y1, x2, y2 = bbox
    box_poly = [(x1,y1), (x2,y1), (x2,y2), (x1,y2)]
    cx, cy = (x1+x2)/2, (y1+y2)/2
    if not point_in_polygon(cx, cy, roi):
        return False, 0.0, "闸门1: 质心不在 ROI 内"
    inter = clip_polygon(box_poly, roi)
    ratio = polygon_area(inter) / polygon_area(roi) if inter else 0.0
    if ratio < min_area_ratio:
        return False, ratio, f"闸门2: 面积比 {ratio:.4f} < {min_area_ratio}"
    return True, ratio, "通过"

# ROI：商场消防通道（画面右下角一块凸四边形）
ROI = [(0.45, 0.45), (0.92, 0.40), (0.95, 0.90), (0.50, 0.95)]

# 模拟 6 个检测结果（零样本模型低阈值放进来的，含大量误报）
DETECTIONS = [
    ("购物车A(真障碍)",   (0.55, 0.55, 0.80, 0.85), 0.31),
    ("远处路人(误报)",     (0.10, 0.10, 0.18, 0.35), 0.28),
    ("海报图案(误报)",     (0.47, 0.47, 0.52, 0.49), 0.29),  # 质心在 ROI 内但太小 → 闸门2
    ("手推车B(真障碍)",    (0.70, 0.50, 0.90, 0.80), 0.33),
    ("广告牌边缘(误报)",   (0.30, 0.35, 0.48, 0.55), 0.27),  # 与 ROI 有重叠但质心在外 → 闸门1
    ("纸箱(真障碍,小)",    (0.60, 0.80, 0.66, 0.89), 0.30),
]

roi_area = polygon_area(ROI)
print(f"ROI 面积 = {roi_area:.3f}（占画面 {roi_area*100:.0f}%）\n")
survivors = []
for name, bbox, conf in DETECTIONS:
    ok, ratio, why = yolo_world_gates(bbox, ROI)
    print(f"{name:16s} conf={conf:.2f} → {'✅ 保留' if ok else '❌ 拦截'}（{why}）")
    if ok: survivors.append(name)
print(f"\n进入规则引擎的检测: {len(survivors)}/{len(DETECTIONS)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.set_title("ROI 几何闸门：6 个低阈值检测的去留", fontsize=13)

# 画面 + ROI
ax.add_patch(plt.Rectangle((0,0), 1, 1, fill=False, ec="gray", lw=1))
roi_patch = plt.Polygon(ROI, alpha=0.15, fc="tab:blue", ec="tab:blue", lw=2)
ax.add_patch(roi_patch)
ax.annotate("ROI: 消防通道", (0.70, 0.935), fontsize=10, color="tab:blue", ha="center")

for name, bbox, conf in DETECTIONS:
    ok, ratio, why = yolo_world_gates(bbox, ROI)
    x1, y1, x2, y2 = bbox
    color = "tab:green" if ok else "tab:red"
    ax.add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1, fill=False, ec=color, lw=2, ls="-" if ok else "--"))
    cx, cy = (x1+x2)/2, (y1+y2)/2
    ax.plot(cx, cy, "o", color=color, ms=4)
    ax.annotate(name.split("(")[0], (x2, y2), fontsize=8, color=color,
                xytext=(4, 4), textcoords="offset points")

ax.set_xlim(-0.05, 1.1); ax.set_ylim(-0.02, 1.05)
ax.set_aspect("equal"); ax.axis("off")
plt.tight_layout(); plt.savefig("/tmp/w12d3_roi.png", dpi=110); plt.show()

## 实验 3：误报率是系统属性——四层兜底的级联效应

域知识.md 说"误报率是市场落地难度大的核心原因"。零样本模型本身精度有限，
但 `min_confidence 0.25 → ROI 几何 → 规则引擎 → CooldownTracker` 四层兜底后，
**系统级误报率**可以远好于模型级。用蒙特卡洛量化这个过程。

同时对比：为什么火灾检测器敢用 0.5 的高阈值（微调模型分数分布分离度不同）。

In [ ]:
N_TP, N_FP = 400, 600   # 一天巡检中：真障碍出现 400 次、原始误报源 600 次

def simulate_day(tp_alpha, tp_beta, fp_alpha, fp_beta):
    """采样一天的检测分数。零样本模型 FP 分布更肥尾（分数更高）。"""
    tp = rng.beta(tp_alpha, tp_beta, N_TP)
    fp = rng.beta(fp_alpha, fp_beta, N_FP)
    return tp, fp

def pr_at_threshold(tp, fp, thr):
    tp_hit = (tp >= thr).sum(); fp_hit = (fp >= thr).sum()
    prec = tp_hit / max(tp_hit + fp_hit, 1)
    rec = tp_hit / N_TP
    return prec, rec

# 零样本 YOLO-World：TP/FP 分离度一般
world_tp, world_fp = simulate_day(3.5, 2.5, 1.6, 4.0)
# 微调 D-Fire：TP/FP 分离度好（专用数据集训练的效果）
fire_tp, fire_fp = simulate_day(9.0, 1.4, 1.0, 8.0)

print("同一阈值下两种模型分数分布的差异：")
for thr in (0.25, 0.50):
    wp, wr = pr_at_threshold(world_tp, world_fp, thr)
    fp_, fr = pr_at_threshold(fire_tp, fire_fp, thr)
    print(f"  阈值 {thr:.2f}: YOLO-World precision={wp:.2f} recall={wr:.2f} | "
          f"D-Fire precision={fp_:.2f} recall={fr:.2f}")
print()
print("→ YOLO-World 阈值从 0.25 提到 0.50：precision 升但 recall 崩（零样本 TP 分数本身不高）")
print("→ 所以代码里选 0.25 低阈值先保召回，把精度问题交给下游管线——这就是'模型弱管线强'")

In [ ]:
# 四层兜底级联模拟（对 YOLO-World @ thr=0.25 的 600 个原始误报源）
thr = 0.25
fp_alive = int((world_fp >= thr).sum())
tp_alive = int((world_tp >= thr).sum())
stages = [("原始检测 @0.25", tp_alive, fp_alive)]

# 层1: ROI 几何——障碍物只会出现在 ROI（本实验约占画面 23%），误报画面均匀分布
#      → FP 只按 ROI 面积比例存活
f_roi = polygon_area(ROI)
fp1 = int(rng.binomial(fp_alive, f_roi)); tp1 = int(tp_alive * 0.97)   # ROI 命中真障碍 97%
stages.append(("+ 闸门1/2: ROI+面积比", tp1, fp1))

# 层2: 规则引擎——需连续/多帧或满足场景规则才产生事件（假设再砍 60% FP, 5% TP 被规则误杀）
fp2 = int(fp1 * 0.4); tp2 = int(tp1 * 0.95)
stages.append(("+ 规则引擎", tp2, fp2))

# 层3: CooldownTracker——同位置 N 小时不重复告警（静态误报一天最多告 1 次）
fp3 = int(fp2 * 0.5); tp3 = int(tp2 * 0.95)   # 真障碍也不该重复告警
stages.append(("+ Cooldown 4h", tp3, fp3))

labels = [s[0] for s in stages]
tps = np.array([s[1] for s in stages]); fps = np.array([s[2] for s in stages])
precisions = tps / np.maximum(tps + fps, 1)

print(f"{'阶段':<22s} {'TP':>4s} {'FP':>4s} {'Precision':>9s}")
for l, t, f, p in zip(labels, tps, fps, precisions):
    print(f"{l:<24s}{t:>4d}{f:>4d}{p:>9.2f}")
print(f"\n最终告警级 recall = {tp3/N_TP:.2f}（告警去重后仍有 {tp3} 条有效告警）")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist([world_tp, world_fp], bins=40, label=["真障碍分数", "误报分数"], color=["tab:green", "tab:red"], alpha=0.7)
for thr_v, name in [(0.25, "world@0.25"), (0.50, "fire@0.50")]:
    axes[0].axvline(thr_v, ls="--", lw=1.5)
    axes[0].annotate(name, (thr_v, axes[0].get_ylim()[1]*0.9), fontsize=9, rotation=90, ha="right")
axes[0].set_title("零样本模型分数分布与阈值选择"); axes[0].legend(); axes[0].set_xlabel("confidence")

x = np.arange(len(labels))
axes[1].bar(x - 0.18, tps, 0.36, label="有效告警(TP)", color="tab:green")
axes[1].bar(x + 0.18, fps, 0.36, label="误报告警(FP)", color="tab:red")
for i, p in enumerate(precisions):
    axes[1].annotate(f"P={p:.2f}", (i, max(tps[i], fps[i]) + 12), ha="center", fontsize=9)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, fontsize=9)
axes[1].set_title("四层兜底级联：误报率是系统属性"); axes[1].legend()
plt.tight_layout(); plt.savefig("/tmp/w12d3_cascade.png", dpi=110); plt.show()

print("\n结论：模型 precision ≈", f"{pr_at_threshold(world_tp, world_fp, 0.25)[0]:.2f}",
      "→ 系统 precision ≈", f"{precisions[-1]:.2f}",
      "。管线把零样本模型救成了可运营系统。")

## 总结：三个实验对应三个选型论断

| 论断（md 第 7 段选型矩阵） | 实验 | 数值证据 |
|---|---|---|
| 闭集方案出局：词表错配是结构性的 | 实验 1 | 购物车/交通锥/梯子 → COCO 只能错标成 suitcase/chair |
| 零样本精度不足用几何管线补偿 | 实验 2 | 6 个低阈值检测只剩 3 个进入规则引擎 |
| 误报率是系统属性，不是模型属性 | 实验 3 | precision 从模型级 ~0.6 → 系统级 ~0.95 |

**为什么是 YOLO-World？** 实验 1 证明只有它把"检测什么"变成数据（实验 1 第二格：
换词表 = 换候选矩阵，模型不动）；实验 2+3 证明它的精度短板可以被管线补；
而 GroundingDINO 在 CPU 上买不起的那部分算力，没有任何管线能补。